# Automatic Crime Detection — Full Training Pipeline

Notebook-kani wuxuu isku darayaa:
- **Pipeline production** (no leakage, tuning, test, save API models)
- **EDA muhiimka ah** ee laga soo qaatay model sax.ipynb (duplicates, wordclouds, top words, before/after cleaning)

Qaybaha:
1. Data loading & cleaning (+ sax duplicate analysis)
2. Preprocessing wadaag ah (stopwords sax + cleaning sax)
3. Visualizations / EDA (sax + charts cad)
4. Train / Val / Test split
5. Feature engineering
6. Baseline training
7. Feature optimization
8. Hyperparameter tuning
9. Hold-out testing
10. Save crime_model.pkl


In [1]:
import sys
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import joblib

# Jupyter's working directory may be the repository root or model/.
_search_roots = (Path.cwd(), *Path.cwd().parents)
AI_MODEL_DIR = next(
    (root / 'ai-model' for root in _search_roots
     if (root / 'ai-model' / 'preprocessing.py').is_file()),
    None,
)
if AI_MODEL_DIR is None:
    raise FileNotFoundError(
        "Could not locate ai-model/preprocessing.py. Start Jupyter from the "
        "crimedetection-sytem repository (or one of its subdirectories)."
    )
AI_MODEL_DIR = AI_MODEL_DIR.resolve()
sys.path.insert(0, str(AI_MODEL_DIR))
from preprocessing import preprocess_text, clean_text, load_somali_stopwords

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

RANDOM_STATE = 42
CRIME_LABEL = 'crime-related'
NON_CRIME_LABEL = 'not crime-related'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup OK')
print('Somali stopwords (from model sax + extras):', len(load_somali_stopwords()))
print('AI model dir:', AI_MODEL_DIR)


ModuleNotFoundError: No module named 'pandas'

## 1. Data Loading & Cleaning

Cleaning-ku wuxuu ka mid yahay: label normalize, missing, **duplicates** (sida model sax), short-text filter.


In [2]:
dataset_path = Path('dataset.csv.csv')
try:
    df = pd.read_csv(dataset_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(dataset_path, encoding='latin1')

df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df = df.dropna(subset=['text', 'category']).copy()
df['text'] = df['text'].astype(str)
df['category'] = df['category'].astype(str).str.strip().str.lower()

# Label cleaning (same idea as model sax)
df['category'] = df['category'].replace({
    'crime': CRIME_LABEL,
    'crime related': CRIME_LABEL,
    'not crime': NON_CRIME_LABEL,
    'not crime related': NON_CRIME_LABEL,
})
df = df[df['category'].isin([CRIME_LABEL, NON_CRIME_LABEL])].copy()
df['text'] = df['text'].str.strip()

print('Rows loaded:', len(df))
print('Missing text:', df['text'].isna().sum())
print('Category counts:\n', df['category'].value_counts())


NameError: name 'pd' is not defined

In [ ]:
# --- From model sax: Duplicate analysis ---
n_before = len(df)
n_dupes = int(df.duplicated(subset=['text']).sum())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Duplicates', 'Unique texts'], [n_dupes, n_before - n_dupes],
       color=['#e74c3c', '#27ae60'], edgecolor='black')
ax.set_title('Duplicate Analysis (from model sax)')
ax.set_ylabel('Count')
for i, v in enumerate([n_dupes, n_before - n_dupes]):
    ax.text(i, v + max(n_before * 0.01, 5), str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '00_duplicates.png', dpi=200, bbox_inches='tight')
plt.show()

df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
df = df[df['text'].str.len() >= 40].copy()

print(f'Removed {n_dupes} duplicate texts | Remaining: {len(df)}')


In [ ]:
# Shared preprocessing (sax stopwords + sax cleaning rules inside preprocessing.py)
df['cleaned_text'] = df['text'].apply(clean_text)
df['preprocessed_text'] = df['text'].apply(preprocess_text)
df = df[df['preprocessed_text'].str.len() >= 20].reset_index(drop=True)

df['text_length'] = df['text'].str.len()
df['word_count'] = df['preprocessed_text'].str.split().str.len()
df['sentence_length'] = df['cleaned_text'].str.len()  # sax naming

print(f'Documents after full cleaning: {len(df)}')
print(df['category'].value_counts())
print('\n--- BEFORE vs AFTER (sample 0, like model sax) ---')
print('BEFORE:', df['text'].iloc[0][:220], '...')
print('AFTER :', df['preprocessed_text'].iloc[0][:220], '...')


## 2. Visualizations & EDA

Waxyaabaha muhiimka ee laga soo qaatay **model sax** + charts pipeline-ka.


In [ ]:
# Category balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['category'].value_counts()
colors = ['#c0392b', '#2980b9']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Qeybinta Categories (tirada documents)')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Tirada (count)')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.02, 0.02))
axes[1].set_title('Saamiga Categories (%)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_category_balance.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Text length + crime vs not-crime averages (model sax comparison)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cat, color in zip([CRIME_LABEL, NON_CRIME_LABEL], ['#c0392b', '#2980b9']):
    subset = df.loc[df['category'] == cat, 'word_count']
    axes[0].hist(subset, bins=35, alpha=0.55, label=cat, color=color, edgecolor='black')
axes[0].set_title('Word Count Distribution by Category')
axes[0].set_xlabel('Words (after preprocess)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

crime_df = df[df['category'] == CRIME_LABEL]
notcrime_df = df[df['category'] == NON_CRIME_LABEL]
comparison = pd.DataFrame({
    'Crime': [crime_df['sentence_length'].mean(), crime_df['word_count'].mean()],
    'Not Crime': [notcrime_df['sentence_length'].mean(), notcrime_df['word_count'].mean()],
}, index=['Sentence Length', 'Word Count'])
comparison.plot(kind='bar', ax=axes[1], color=['#c0392b', '#2980b9'], edgecolor='black')
axes[1].set_title('Crime vs Not Crime Averages (from model sax)')
axes[1].set_ylabel('Average')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_text_length_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print(comparison.round(1))


In [ ]:
# Top words per class (sax + side-by-side)
def top_words(texts, n=15):
    counter = Counter()
    for t in texts:
        counter.update(str(t).split())
    return counter.most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, cat, color in zip(axes, [CRIME_LABEL, NON_CRIME_LABEL], ['#c0392b', '#2980b9']):
    words = top_words(df.loc[df['category'] == cat, 'preprocessed_text'], 15)
    labels = [w for w, _ in words][::-1]
    vals = [c for _, c in words][::-1]
    ax.barh(labels, vals, color=color, edgecolor='black')
    ax.set_title(f'Top 15 words — {cat}')
    ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_top_words_by_class.png', dpi=200, bbox_inches='tight')
plt.show()

print('Top crime words:', top_words(crime_df['preprocessed_text'], 10))
print('Top not-crime words:', top_words(notcrime_df['preprocessed_text'], 10))


In [ ]:
# Overall most common words + word length distribution (from model sax)
all_words = ' '.join(df['preprocessed_text']).split()
word_freq = Counter(all_words)
common_words = word_freq.most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
words = [w for w, _ in common_words]
counts = [c for _, c in common_words]
axes[0].bar(words, counts, color='#34495e', edgecolor='black')
axes[0].set_title('Most Common Words (all data)')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Frequency')
axes[0].tick_params(axis='x', rotation=55)

word_lengths = [len(w) for w in all_words]
axes[1].hist(word_lengths, bins=20, color='#16a085', edgecolor='black')
axes[1].set_title('Word Length Distribution (from model sax)')
axes[1].set_xlabel('Word length (chars)')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03b_word_freq_and_length.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Word clouds — general / crime / not-crime (from model sax)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

clouds = [
    ('All texts', ' '.join(df['preprocessed_text']), 'viridis'),
    ('Crime-related', ' '.join(crime_df['preprocessed_text']), 'Reds'),
    ('Not crime-related', ' '.join(notcrime_df['preprocessed_text']), 'Blues'),
]
for ax, (title, text, cmap) in zip(axes, clouds):
    if text.strip():
        wc = WordCloud(width=900, height=500, background_color='white',
                       colormap=cmap, max_words=120).generate(text)
        ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03c_wordclouds.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved wordclouds (model sax style)')


## 3. Train / Validation / Test Split (NO leakage)

Split **ka hor** TF-IDF — ka duwan model sax oo fit_transform sameeyay data-da oo dhan.


In [ ]:
X_text = df['preprocessed_text']
y = df['category']

X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', len(X_train_text), y_train.value_counts().to_dict())
print('Val  :', len(X_val_text), y_val.value_counts().to_dict())
print('Test :', len(X_test_text), y_test.value_counts().to_dict())


## 4. Feature Engineering (fit on TRAIN only)


In [ ]:
def make_vectorizer(max_features=10000, ngram_range=(1, 2), analyzer='word'):
    return TfidfVectorizer(
        max_features=max_features,
        min_df=2,
        max_df=0.92,
        ngram_range=ngram_range,
        analyzer=analyzer,
        sublinear_tf=True,
    )

vectorizer = make_vectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

print('TF-IDF features:', X_train.shape[1])
print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)


## 5. Model Training (baseline)

Models-ka sida sax + Gradient Boosting / LinearSVC. Metrics waxaa ku jira **Crime Recall**.


In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Naive Bayes': MultinomialNB(alpha=0.5),
    'Decision Tree': DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=25, random_state=RANDOM_STATE,
        class_weight='balanced_subsample', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, random_state=RANDOM_STATE),
    'Linear SVM': LinearSVC(max_iter=3000, class_weight='balanced', dual=False, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=7),
}

def evaluate_model(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {
        'accuracy': accuracy_score(y_eval, pred),
        'precision': precision_score(y_eval, pred, average='weighted', zero_division=0),
        'recall': recall_score(y_eval, pred, average='weighted', zero_division=0),
        'f1': f1_score(y_eval, pred, average='weighted', zero_division=0),
        'crime_recall': recall_score(y_eval, pred, pos_label=CRIME_LABEL, zero_division=0),
        'predictions': pred,
    }

trained = {}
baseline_rows = []

print('Training baseline models...\n')
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    metrics = evaluate_model(model, X_val, y_val)
    baseline_rows.append({
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1'],
        'Crime Recall': metrics['crime_recall'],
    })
    print(f"  {name:22s}  F1={metrics['f1']:.4f}  CrimeRecall={metrics['crime_recall']:.4f}")
    # Per-model report like model sax (validation)
    print(classification_report(y_val, metrics['predictions'], digits=3))

baseline_df = pd.DataFrame(baseline_rows).sort_values('F1', ascending=False).reset_index(drop=True)
print('\\n=== BASELINE VALIDATION ===')
baseline_df


In [ ]:
plot_df = baseline_df.sort_values('F1')
fig, ax = plt.subplots(figsize=(12, 6))
y_pos = np.arange(len(plot_df))
ax.barh(y_pos, plot_df['F1'], color='#27ae60', edgecolor='black', height=0.45, label='F1')
ax.barh(y_pos + 0.45, plot_df['Crime Recall'], color='#e67e22', edgecolor='black', height=0.45, label='Crime Recall')
ax.set_yticks(y_pos + 0.225)
ax.set_yticklabels(plot_df['Model'])
ax.set_xlim(0, 1.05)
ax.set_xlabel('Score (0–1)')
ax.set_title('Baseline Models — Validation F1 vs Crime Recall')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_baseline_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Best baseline:', baseline_df.iloc[0]['Model'])


## 6. Optimization — Feature Configuration Search


In [ ]:
feature_configs = [
    {'name': 'word_3k_(1,1)', 'max_features': 3000, 'ngram_range': (1, 1), 'analyzer': 'word'},
    {'name': 'word_5k_(1,1)', 'max_features': 5000, 'ngram_range': (1, 1), 'analyzer': 'word'},  # model sax default size
    {'name': 'word_10k_(1,2)', 'max_features': 10000, 'ngram_range': (1, 2), 'analyzer': 'word'},
    {'name': 'word_15k_(1,3)', 'max_features': 15000, 'ngram_range': (1, 3), 'analyzer': 'word'},
    {'name': 'char_8k_(3,5)', 'max_features': 8000, 'ngram_range': (3, 5), 'analyzer': 'char_wb'},
]

opt_rows = []
print('Feature optimization...\n')
for cfg in feature_configs:
    vec = make_vectorizer(cfg['max_features'], cfg['ngram_range'], cfg['analyzer'])
    Xt = vec.fit_transform(X_train_text)
    Xv = vec.transform(X_val_text)
    clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    clf.fit(Xt, y_train)
    m = evaluate_model(clf, Xv, y_val)
    opt_rows.append({'Config': cfg['name'], 'Features': Xt.shape[1], 'F1': m['f1'],
                     'Crime Recall': m['crime_recall'], 'Accuracy': m['accuracy']})
    print(f"  {cfg['name']:18s} feats={Xt.shape[1]:5d}  F1={m['f1']:.4f}")

opt_df = pd.DataFrame(opt_rows).sort_values('F1', ascending=False).reset_index(drop=True)
best_feature_name = opt_df.iloc[0]['Config']
print('\\nBest feature config:', best_feature_name)
opt_df


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(opt_df))
width = 0.35
ax.bar(x - width/2, opt_df['F1'], width, label='F1', color='#27ae60', edgecolor='black')
ax.bar(x + width/2, opt_df['Crime Recall'], width, label='Crime Recall', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(opt_df['Config'], rotation=15, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Optimization: TF-IDF Feature Configs (includes sax 5k)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_feature_optimization.png', dpi=200, bbox_inches='tight')
plt.show()

best_cfg = next(c for c in feature_configs if c['name'] == best_feature_name)
vectorizer = make_vectorizer(best_cfg['max_features'], best_cfg['ngram_range'], best_cfg['analyzer'])
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)
print('Using:', best_feature_name, X_train.shape)


## 7. Hyperparameter Tuning (ma jirin model sax — waa cusub)


In [ ]:
from scipy.sparse import vstack

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
X_tune = vstack([X_train, X_val])
y_tune = pd.concat([y_train, y_val], ignore_index=True)

tuning_spaces = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2500, class_weight='balanced', random_state=RANDOM_STATE),
        [
            {'C': np.logspace(-2, 2, 10), 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear', 'saga']},
            {'C': np.logspace(-2, 2, 10), 'penalty': ['l1'], 'solver': ['liblinear', 'saga']},
        ],
    ),
    'Linear SVM': (
        LinearSVC(max_iter=4000, class_weight='balanced', dual=False, random_state=RANDOM_STATE),
        {'C': np.logspace(-2, 2, 12)},
    ),
    'Random Forest': (
        RandomForestClassifier(class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
        {
            'n_estimators': [120, 200, 300],
            'max_depth': [15, 25, 40, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 0.3],
        },
    ),
    'Naive Bayes': (
        MultinomialNB(),
        {'alpha': [0.01, 0.1, 0.5, 1.0, 2.0], 'fit_prior': [True, False]},
    ),
}

tuned_models = {}
tune_rows = []
print('Hyperparameter tuning...\n')
for name, (estimator, params) in tuning_spaces.items():
    search = RandomizedSearchCV(
        estimator, param_distributions=params, n_iter=12, scoring='f1_weighted',
        cv=cv, random_state=RANDOM_STATE, n_jobs=-1, refit=True,
    )
    search.fit(X_tune, y_tune)
    tuned_models[name] = search.best_estimator_
    tune_rows.append({'Model': name, 'Best CV F1': search.best_score_, 'Best Params': str(search.best_params_)})
    print(f"  {name:22s}  CV F1={search.best_score_:.4f}")
    print(f"    params: {search.best_params_}")

tune_df = pd.DataFrame(tune_rows).sort_values('Best CV F1', ascending=False).reset_index(drop=True)
tune_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_t = tune_df.sort_values('Best CV F1')
ax.barh(plot_t['Model'], plot_t['Best CV F1'], color='#8e44ad', edgecolor='black')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Cross-validated F1 (weighted)')
ax.set_title('Hyperparameter Tuning Results (3-fold CV)')
for i, v in enumerate(plot_t['Best CV F1']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_hyperparameter_tuning.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Model Testing (hold-out TEST)


In [ ]:
test_rows = []
test_preds = {}
for name, model in tuned_models.items():
    m = evaluate_model(model, X_test, y_test)
    test_preds[name] = m['predictions']
    test_rows.append({
        'Model': name,
        'Accuracy': m['accuracy'],
        'F1': m['f1'],
        'Crime Precision': precision_score(y_test, m['predictions'], pos_label=CRIME_LABEL, zero_division=0),
        'Crime Recall': m['crime_recall'],
    })

best_baseline_name = baseline_df.iloc[0]['Model']
m_base = evaluate_model(trained[best_baseline_name], X_test, y_test)
test_rows.append({
    'Model': f'{best_baseline_name} (baseline)',
    'Accuracy': m_base['accuracy'],
    'F1': m_base['f1'],
    'Crime Precision': precision_score(y_test, m_base['predictions'], pos_label=CRIME_LABEL, zero_division=0),
    'Crime Recall': m_base['crime_recall'],
})

test_df = pd.DataFrame(test_rows).sort_values('F1', ascending=False).reset_index(drop=True)
print('=== HOLD-OUT TEST ===')
print(test_df.to_string(index=False))

production_name = max(tuned_models.keys(), key=lambda n: f1_score(y_test, test_preds[n], average='weighted'))
production_model = tuned_models[production_name]
print('\\nSelected production model:', production_name)


In [ ]:
y_pred = production_model.predict(X_test)
labels_order = [NON_CRIME_LABEL, CRIME_LABEL]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: NOT crime', 'Pred: CRIME'],
            yticklabels=['True: NOT crime', 'True: CRIME'])
axes[0].set_title(f'Confusion Matrix — {production_name}')

y_bin = (y_test == CRIME_LABEL).astype(int)
if hasattr(production_model, 'predict_proba'):
    scores = production_model.predict_proba(X_test)[:, list(production_model.classes_).index(CRIME_LABEL)]
elif hasattr(production_model, 'decision_function'):
    scores = production_model.decision_function(X_test)
else:
    scores = (y_pred == CRIME_LABEL).astype(float)

fpr, tpr, _ = roc_curve(y_bin, scores)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#c0392b', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (crime-related)')
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_test_confusion_roc.png', dpi=200, bbox_inches='tight')
plt.show()
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_t = test_df.copy()
x = np.arange(len(plot_t))
w = 0.2
ax.bar(x - 1.5*w, plot_t['Accuracy'], w, label='Accuracy', color='#3498db', edgecolor='black')
ax.bar(x - 0.5*w, plot_t['F1'], w, label='F1', color='#27ae60', edgecolor='black')
ax.bar(x + 0.5*w, plot_t['Crime Precision'], w, label='Crime Precision', color='#9b59b6', edgecolor='black')
ax.bar(x + 1.5*w, plot_t['Crime Recall'], w, label='Crime Recall', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(plot_t['Model'], rotation=20, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Testing — Hold-out Test Metrics')
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '08_model_testing.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
fn_mask = (y_test == CRIME_LABEL) & (y_pred != CRIME_LABEL)
fp_mask = (y_test == NON_CRIME_LABEL) & (y_pred == CRIME_LABEL)
print(f'False Negatives (missed crime): {fn_mask.sum()}')
print(f'False Positives (false alarm): {fp_mask.sum()}')
print('\\n--- Sample missed crimes ---')
for i, t in enumerate(X_test_text[fn_mask].head(5), 1):
    print(f'{i}. {t[:180]}...')
print('\\n--- Sample false alarms ---')
for i, t in enumerate(X_test_text[fp_mask].head(5), 1):
    print(f'{i}. {t[:180]}...')


## 9. Save Production Artifacts


In [ ]:
if not hasattr(production_model, 'predict_proba'):
    print('Wrapping with CalibratedClassifierCV...')
    calibrator = CalibratedClassifierCV(production_model, method='sigmoid', cv=3)
    calibrator.fit(X_tune, y_tune)
    production_model = calibrator

model_path = AI_MODEL_DIR / 'crime_model.pkl'
vectorizer_path = AI_MODEL_DIR / 'vectorizer.pkl'
joblib.dump(production_model, model_path)
joblib.dump(vectorizer, vectorizer_path)

meta = {
    'model_name': production_name,
    'feature_config': best_feature_name,
    'n_features': int(X_train.shape[1]),
    'train_size': int(X_train.shape[0]),
    'test_f1': float(f1_score(y_test, production_model.predict(X_test), average='weighted')),
    'test_crime_recall': float(recall_score(y_test, production_model.predict(X_test), pos_label=CRIME_LABEL)),
    'preprocess': 'ai-model/preprocessing.py::preprocess_text',
    'stopwords_source': 'model sax + extras (somali_stopwords.json)',
}
joblib.dump(meta, AI_MODEL_DIR / 'model_meta.pkl')
print('Saved:', model_path)
print('Saved:', vectorizer_path)
print('Meta:', meta)
print('Restart AI API (port 5001) after save.')


## 10. Smoke test (API path)


In [ ]:
samples = [
    'Nin ayaa lagu dilay magaalada Muqdisho ee degmada Hodan habeen hore.',
    'Ciyaaraha football-ka ayaa caawa ka dhacaya garoonka Muqdisho Stadium.',
    'Qarax ayaa ka dhacay suuqa, dad badan ayaa ku dhaawacmay.',
]
loaded_model = joblib.load(AI_MODEL_DIR / 'crime_model.pkl')
loaded_vec = joblib.load(AI_MODEL_DIR / 'vectorizer.pkl')
print('Smoke test:')
for s in samples:
    processed = preprocess_text(s)
    vec = loaded_vec.transform([processed])
    pred = loaded_model.predict(vec)[0]
    conf = max(loaded_model.predict_proba(vec)[0]) * 100 if hasattr(loaded_model, 'predict_proba') else float('nan')
    print(f'  [{pred}] ({conf:.1f}%) :: {s[:70]}')
